# 项目：航空 AI 助手（FlightAI）

## 练习目标（理念）

把第 2 周学过的能力拼成一个 **航空公司客户支持助理**：

- Gradio 聊天界面
- OpenAI **工具调用（tool calling / function calling）**：查票价、查航班状态
- 进阶：**多模态**——DALL·E 出图 + TTS 朗读回复
- 自定义 `gr.Blocks` UI（聊天 + 图 + 音频）

## 和本课第 2 周的关系

| 本课概念 | 本练习里你会看到 |
|----------|------------------|
| Chat Completions + tools | `tools=[...]`、`finish_reason=="tool_calls"` |
| Tool schema（JSON） | `price_function` / `flight_status_function` |
| Gradio ChatInterface / Blocks | 先简单聊天，后多模态自定义布局 |
| 多模态 | `images.generate`（DALL·E）+ `audio.speech`（TTS） |

## 怎么跑

1. `.env` 里配置 `OPENAI_API_KEY`
2. 按顺序运行；工具版聊天前先定义 `handle_tool_calls`
3. 多模态 Blocks 段注意费用：图像生成按次计费；`auth` 账号密码以原代码为准


In [ ]:
# ========== 导入：环境、JSON、OpenAI、Gradio、SQLite ==========

# 标准库 os：读环境变量
import os
# 标准库 json：解析 tool_call.function.arguments，以及把状态 dict 序列化回模型
import json
# load_dotenv：从 .env 加载密钥
from dotenv import load_dotenv
# OpenAI 客户端：聊天、工具调用、出图、TTS
from openai import OpenAI
# Gradio：快速搭 Web 聊天 / Blocks UI
import gradio as gr
# sqlite3：本练习导入了（可能配合 prices.db）；当前可见逻辑主要用内存字典
import sqlite3


In [ ]:
# ========== 初始化：密钥检查、模型常量、客户端、DB 路径 ==========

# override=True：.env 覆盖已有环境变量
load_dotenv(override=True)

# 读取 OpenAI 密钥并打印前缀自检
openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
# 本练习选用的聊天模型 id（字符串勿改译）
MODEL = "gpt-4.1-mini"
# 创建 OpenAI 客户端（默认读 OPENAI_API_KEY）
openai = OpenAI()

# SQLite 数据库文件名常量（与课程示例一致；本格只赋值）
DB = "prices.db"


In [ ]:
# ========== 系统提示：航空客服人设（英文保持原样）==========

# system_message 会作为 messages 里 role=system 的内容；改译会改变助手行为
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, friendly and courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer after checking all available tools, say so.
Use the provided functions to answer questions about ticket prices and flight status before responding.
"""


In [ ]:
# ========== Tool Schema：告诉模型「可以调哪些函数」==========

# get_ticket_price 的 JSON 函数规范（OpenAI tools 的 function 字段）
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

# get_flight_status 的 JSON 函数规范
flight_status_function = {
    "name": "get_flight_status",
    "description": "Check the status of a booked flight by flight number and date.",
    "parameters": {
        "type": "object",
        "properties": {
            "flight_number": {
                "type": "string",
                "description": "The flight number to check status for",
            },
            "date": {
                "type": "string",
                "description": "The date of the flight (YYYY-MM-DD)",
            },
        },
        "required": ["flight_number", "date"],
        "additionalProperties": False
    }
}

# 更新模型使用工具：包装成 OpenAI tools 列表（type=function）
tools = [
    {"type": "function", "function": price_function},
    {"type": "function", "function": flight_status_function}
]
# 单元格末尾写 tools：在 Jupyter 里直接展示这个列表
tools


In [ ]:
# ========== 工具实现：内存票价表 + get_ticket_price ==========

# 目的地城市（小写 key）→ 票价字符串；查不到走默认文案
ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    # 打印日志：方便在终端看到「模型真的调了工具」
    print(f"Tool called for city {destination_city}")
    # lower() 统一大小写；dict.get 第二参是缺省值
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    # 返回给人/给模型看的英文句子（影响回复，保持英文）
    return f"The price of a ticket to {destination_city} is {price}"


In [ ]:
# ========== 模拟航班状态表（内存列表）==========

# 每项是 {航班号: {date, status}}；后面 get_flight_status 会线性查找
flights_record_status = [
    {"LN123": {"date": "2026-03-05", "status": "On Time"}},
    {"PR456": {"date": "2026-03-12", "status": "Delayed"}},
    {"TK789": {"date": "2026-03-18", "status": "Cancelled"}},
    {"BL321": {"date": "2026-03-22", "status": "On Time"}}
]
# 展示列表内容，方便肉眼核对
flights_record_status


In [ ]:
# ========== 工具实现：按航班号查状态 ==========

def get_flight_status(flight_number: str) -> dict:
    """Fetches the status of a flight given its flight number from flights_record_status."""
    # 遍历模拟表；key 命中则取出内层 dict
    for flight in flights_record_status:
        if flight_number in flight:
            flight_info = flight[flight_number]
            # 只返回航班号 + 状态（原逻辑未把 date 带回）
            return {
                'flight_number': flight_number,
                'status': flight_info['status']
            }
    # 如果没有找到，则返回未知状态
    return {
        'flight_number': flight_number,
        'status': 'Unknown'
    }


In [ ]:
# ========== 简易规则助手（占位）：演示「状态 vs 票价」分支 ==========

# 将 get_flight_status 集成到 Flight Assistant 逻辑中

def flight_assistant(user_input: str):
    """
    Handles user requests for booking flights or checking flight status.
    """
    # 用户话里含 status → 走查状态；否则走查票价（提取逻辑仍是占位常量）
    if "status" in user_input.lower():
        # 从用户输入中提取航班号（占位符逻辑）
        flight_number = "LN123"  # Example, replace with extraction logic
        return get_flight_status(flight_number)
    else:
        # 从用户输入中提取目的地（占位符逻辑）
        destination = "Paris"  # Example, replace with extraction logic
        return get_ticket_price(destination)


In [ ]:
# ========== 无工具版聊天：仅 system + 历史 + 用户消息 ==========

def chat(message, history):
    # Gradio messages 格式 → 只保留 role/content，去掉多余字段
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    # system 在前，再拼历史，最后追加本轮 user
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 本格故意不传 tools：先验证「纯聊天」通路
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

# ChatInterface：标准聊天 UI；fn=chat 收到 (message, history)
gr.ChatInterface(fn=chat).launch()


In [ ]:
# ========== 带工具版 chat：tool_calls 循环直到模型说完 ==========

def chat(message, history):
    # 规范化 history 为 OpenAI messages 形状
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # system + 历史 + 本轮用户
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    # 这次传入 tools，模型可以发起 function calling
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 只要 finish_reason 是 tool_calls，就本地执行工具，再把结果塞回 messages 继续问模型
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        # 先把助手那条（含 tool_calls）追加进对话
        messages.append(message)
        # 再追加各 tool 角色的结果
        messages.extend(responses)
        # 带着工具结果再次请求，直到不再要工具
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    # 最终自然语言回复
    return response.choices[0].message.content


In [ ]:
# ========== handle_tool_calls：把模型点名的函数真正执行掉 ==========

def handle_tool_calls(message):
    # 收集多条 tool 角色消息（一轮可能并行多个 tool_call）
    responses = []
    for tool_call in message.tool_calls:
        # 分支 1：查票价
        if tool_call.function.name == "get_ticket_price":
            # arguments 是 JSON 字符串 → dict
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
        # 分支 2：查航班状态
        elif tool_call.function.name == "get_flight_status":
            arguments = json.loads(tool_call.function.arguments)
            flight_number = arguments.get('flight_number')
            status_details = get_flight_status(flight_number)
            responses.append({
                "role": "tool",
                # 状态是 dict，序列化成 JSON 字符串再回传
                "content": json.dumps(status_details),
                "tool_call_id": tool_call.id
            })
    return responses


In [ ]:
# ========== 启动带工具的 ChatInterface ==========

# 注意：需先定义上面的 chat / handle_tool_calls，再 launch
gr.ChatInterface(fn=chat).launch()


## Gradio 实际在干什么？

1. Gradio 根据你用 Python 描述的 UI，生成一个前端（基于 **Svelte**）应用
2. Gradio 再启动一个基于 **Starlette** 的服务器，在空闲端口上托管这个前端
3. 为你的回调（例如 `chat()`）创建后端路由；点「提交」就会打到正确的路由

前端与后端的接线由 Gradio 自动完成——写法简单，效果却像魔法。


# 多模态（Multimodal）进阶

我们可以用 GPT-4o 背后的图像生成模型 **DALL·E-3** 来做配图。

下面把它封装成名为 `artist` 的函数。

### 价格提醒

每次生成图像大约 **4 美分**——别沉迷狂刷图！


In [ ]:
# ========== 图像处理相关导入 ==========

# Some 导入 for handling images
# base64：DALL·E 返回 b64_json 时要解码
import base64
# BytesIO：把字节当成「内存文件」交给 PIL 打开
from io import BytesIO
# PIL.Image：在笔记本里显示 / 作为 Gradio Image 输出
from PIL import Image


In [ ]:
# ========== artist：按城市名生成度假风格配图 ==========

def artist(city):
    # images.generate：调用 DALL·E-3；prompt 里的英文模板保持原样
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    # 取出第一张图的 Base64 载荷
    image_base64 = image_response.data[0].b64_json
    # Base64 字符串 → 原始字节
    image_data = base64.b64decode(image_base64)
    # 字节流 → PIL Image 对象
    return Image.open(BytesIO(image_data))


In [ ]:
# ========== 试跑：给 New York City 出一张图并 display ==========

image = artist("New York City")
display(image)


In [ ]:
# ========== talker：把文本变成语音（TTS）==========

def talker(message):
    # audio.speech.create：文本转语音；voice 可换成 alloy / coral 等
    response = openai.audio.speech.create(
      model="gpt-4o-mini-tts",
      voice="onyx",    # Also, try replacing onyx with alloy or coral
      input=message
    )
    # .content 是音频字节，后面交给 Gradio Audio
    return response.content


## 把能力带回家

1. **多模态** AI 助手：既能出图也能出声
2. **工具调用** + 数据查找（票价 / 航班状态）
3. 向 **Agentic** 工作流迈出一步：模型决定何时调工具、何时回复用户


In [ ]:
# ========== 多模态 chat：工具循环 + TTS + 可选出图 ==========

def chat(history):
    # Blocks 版：history 已是消息列表；规范化 role/content
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    # system + 完整历史（本轮用户已在 history 里）
    messages = [{"role": "system", "content": system_message}] + history
    # 带 tools 发起请求
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    # cities：从票价工具参数里收集目的地，供后面出图
    cities = []
    image = None

    # 工具循环：执行 → 回填 → 再问模型
    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    # 最终文本回复
    reply = response.choices[0].message.content
    # 把助手消息写回 Gradio history
    history += [{"role":"assistant", "content":reply}]

    # TTS：把回复念出来
    voice = talker(reply)

    # 若本轮查过票价城市，用第一个城市出一张度假图
    if cities:
        image = artist(cities[0])
    
    # 返回：更新后的 history、音频字节、可选图片
    return history, voice, image


In [ ]:
# ========== 多模态专用工具处理：顺便收集城市名 ==========

def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        # 本函数只处理票价工具（原逻辑如此）
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            # 记录城市，供 artist() 出图
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    # 同时返回 tool 消息列表与城市列表
    return responses, cities


## Gradio UI 的三种类型

- `gr.Interface`：标准、简单的输入→输出 UI
- `gr.ChatInterface`：开箱即用的聊天机器人 UI
- `gr.Blocks`：自定义布局，你自己摆组件、接线回调


In [ ]:
# ========== Blocks UI：聊天 + 图 + 音频 + 提交链路 ==========

# 回调（以及上面的 chat() 函数）

def put_message_in_chatbot(message, history):
        # 清空输入框，并把用户消息追加进 chatbot history
        return "", history + [{"role":"user", "content":message}]

# 用户界面定义：两列（聊天 | 图）+ 音频行 + 输入行

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# 将事件连接到回调：先塞用户消息，再 .then 跑多模态 chat

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output]
    )

# inbrowser=True 自动开浏览器；auth 为简易登录（用户名/密码保持原样）
ui.launch(inbrowser=True, auth=("ed", "bananas"))


# 练习与商业应用

- **加更多工具**：例如模拟真正预订航班。已有同学在 community-contributions 里放了示例。
- **下一步**：搬到你自己的业务——用「能干活的工具」做一个多模态助手：客服？新人入职？可能性很多。
- 也可对照单独笔记本里的 **第 2 周周末练习**。


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="宽度：150px；高度：150px；垂直对齐：中间；">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">我对你有一个特殊请求</h2>
            <span style="color:#090;">
                我的编辑告诉我：学生在 Udemy 上给这门课打分影响很大——这是 Udemy 决定是否向更多人展示课程的主要信号之一。若你愿意花几分钟评价，我会非常感激！若需要帮助，也欢迎随时邮件联系 ed@edwarddonner.com。
            </span>
        </td>
    </tr>
</table>
